<a href="https://colab.research.google.com/github/DeepFluxion/Mack_2026_Data_Science_Expirience/blob/main/notebooks/bank_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Pré-processamento de Dados — Bank Marketing UCI
## Módulo 2: Data Preparation para Modelos Preditivos

> **Dataset:** UCI Bank Marketing (`bank-additional-full.csv`) — 41.188 registros · 20 features + target
> **Objetivo:** Preparar os dados para modelagem de propensão à adesão de depósito a prazo
> **Nível:** MBA — Data Science | **Linguagem:** Python 3.x + scikit-learn

---

### 📋 Estrutura do Notebook

| # | Seção | Conteúdo |
|---|-------|----------|
| 1 | Teoria do pré-processamento | Conceitos: missing data, encoding, scaling, data leakage |
| 2 | Estruturas Sklearn | Pipeline, ColumnTransformer, transformadores customizados |
| 3 | Análise variável a variável | Diagnóstico e decisão técnica para cada feature |
| 4 | Geração do dataset final | Pipeline completa, split, validação e exportação |
| 5 | Conclusões | Tabela-resumo de decisões e próximos passos |

> ⚠️ **Alerta de Data Leakage:** A variável `duration` (duração da ligação em segundos) é altamente correlacionada com o target `y`, mas **só é conhecida após a ligação ocorrer**. Ela será descartada em todas as etapas preditivas.

In [ ]:
# ══════════════════════════════════════════════════════
# CÉLULA 1 — Detecção de ambiente e dependências
# ══════════════════════════════════════════════════════
import sys

# Detecta se está no Google Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("Ambiente: Google Colab detectado")
except ImportError:
    IN_COLAB = False
    print("Ambiente: Execucao local / Jupyter detectado")

# No Colab, instala dependências caso necessário
if IN_COLAB:
    print("Instalando dependencias (pode levar alguns segundos)...")
    import subprocess
    pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
    print("Dependencias OK")

In [ ]:
# ══════════════════════════════════════════════════════
# CÉLULA 2 — Imports
# ══════════════════════════════════════════════════════
import io
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Sklearn — Preprocessing
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OrdinalEncoder, OneHotEncoder, LabelEncoder,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # obrigatorio antes do import
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11
})

SEED = 42
print("Imports realizados com sucesso")

---
## 📖 Seção 1 — Teoria do Pré-processamento

> *"Garbage in, garbage out"* — o desempenho de qualquer modelo de Machine Learning depende diretamente da qualidade dos dados que o alimentam. O pré-processamento não é uma etapa opcional: é a fundação sobre a qual toda modelagem é construída.

### 1.1 — Por que pré-processar?

A maioria dos algoritmos de ML opera sobre **matrizes numéricas densas** e sem valores ausentes. Dados do mundo real raramente chegam nesse formato.

| Problema nos dados brutos | Consequência sem tratamento | Solução |
|---------------------------|----------------------------|---------|
| Valores faltantes | Erros em tempo de execução ou distorção nos resultados | Imputação ou remoção |
| Escalas muito distintas | Algoritmos baseados em distância (KNN, SVM, regressão) são dominados pelas variáveis de maior magnitude | Normalização / Padronização |
| Variáveis categóricas (strings) | A maioria dos algoritmos não aceita texto como entrada | Encoding numérico |
| Distribuições assimétricas | Gradientes instáveis e viés em imputações | Transformação logarítmica |
| Data leakage | Métricas infladas, modelo inútil em produção | Isolamento rigoroso de treino e teste |

**Desafios presentes neste dataset:**
- `unknown` codificado em 6 variáveis categóricas (missing implícito)
- `duration`: variável com **vazamento de dados** → remoção obrigatória
- `pdays`: valor 999 = semântica especial ("nunca contatado"), não um número de dias real
- `month` e `day_of_week`: variáveis temporais cíclicas que exigem encoding especial
- Variáveis numéricas com distribuições muito distintas entre si

### 1.2 — Dados faltantes: tipos e estratégias de imputação

#### Taxonomia de Rubin (1976)

**MCAR — Missing Completely At Random**
A ausência não tem relação com nenhuma variável observada ou não observada.
*Exemplo:* Falha aleatória de sensor no registro de dados.
*Estratégia:* Qualquer técnica funciona; remoção é aceitável se a proporção for pequena.

**MAR — Missing At Random**
A ausência está relacionada a outras variáveis observadas, mas não ao próprio valor faltante.
*Exemplo:* Clientes mais jovens tendem a não informar estado civil.
*Estratégia:* Imputação condicional (KNN, modelos preditivos, MICE).

**MNAR — Missing Not At Random**
A ausência está relacionada ao próprio valor não observado.
*Exemplo:* `default = 'unknown'` — clientes inadimplentes têm incentivo para omitir essa informação.
*Estratégia:* Manter como categoria separada ou modelar a ausência explicitamente.

---

#### Estratégias disponíveis no scikit-learn

```python
SimpleImputer(strategy='mean')           # média        — numéricas simétricas
SimpleImputer(strategy='median')         # mediana       — numéricas com outliers
SimpleImputer(strategy='most_frequent')  # moda          — categóricas
SimpleImputer(strategy='constant', fill_value='MISSING')  # valor constante

KNNImputer(n_neighbors=5)               # k vizinhos mais próximos
IterativeImputer(max_iter=10)           # MICE: imputação múltipla por equações encadeadas
```

> 📌 **Regra fundamental:** o `fit()` do imputer deve ocorrer **apenas no conjunto de treino**.

### 1.3 — Encoding de variáveis categóricas

Algoritmos de ML não processam strings. Precisamos converter categorias em números que preservem (ou não introduzam) relações artificiais.

#### Ordinal Encoding (com ordem explícita)
Mapeia categorias a inteiros respeitando uma **hierarquia real** pré-definida.
*Aplicação:* `education` — `basic.4y < basic.6y < ... < university.degree`

```python
OrdinalEncoder(categories=[['basic.4y', 'basic.6y', 'basic.9y',
                             'high.school', 'professional.course',
                             'university.degree', 'unknown']])
```

#### One Hot Encoding (OHE)
Cria uma coluna binária para cada categoria. Ideal para variáveis **nominais** (sem ordem natural).
Use `drop='first'` para evitar multicolinearidade — a *dummy trap*.

#### Cyclical Encoding (sin/cos)
Para variáveis com **periodicidade circular**, OHE perde a continuidade temporal:
dezembro está próximo de janeiro, mas OHE os trata como totalmente independentes.

$$\text{sin\_month} = \sin\!\left(\frac{2\pi \cdot \text{month}}{12}\right) \quad
  \text{cos\_month} = \cos\!\left(\frac{2\pi \cdot \text{month}}{12}\right)$$

#### Target Encoding
Substitui a categoria pela média do target naquela categoria. Muito poderoso, mas sujeito a leakage — requer validação cruzada. **Não utilizado neste notebook.**

---

| Variável | Encoding escolhido | Justificativa |
|----------|--------------------|---------------|
| `education` | OrdinalEncoder (ordem explícita) | Hierarquia educacional mensurável |
| `month`, `day_of_week` | Cyclical sin/cos | Periodicidade circular |
| `job`, `marital`, `contact`, `poutcome` | OHE | Variáveis nominais sem ordem |
| `housing`, `loan`, `default` | OHE (com tratamentos distintos de missing) | Poucas categorias, nominal |

### 1.4 — Escalonamento de variáveis numéricas

Algoritmos sensíveis à escala (regressão logística, SVM, KNN, redes neurais) exigem que as features estejam em magnitudes comparáveis.

#### StandardScaler — Padronização Z-score

$$z = \frac{x - \mu}{\sigma}$$

Distribui os dados com **média 0 e desvio padrão 1**. Sensível a outliers (usa média e desvio).
Adequado para distribuições aproximadamente normais.

#### MinMaxScaler — Normalização min-max

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Comprime os valores para o intervalo **[0, 1]**. Extremamente sensível a outliers.
Adequado quando o intervalo precisa ser preservado (ex: pixels de imagem).

#### RobustScaler — Escalonamento robusto via IQR

$$x' = \frac{x - Q_2}{Q_3 - Q_1}$$

Usa **mediana** e **IQR** em vez de média e desvio padrão. Resistente a outliers.
Ideal para variáveis com caudas pesadas ou outliers legítimos (ex: `campaign`, `previous`).

---

| Variável | Distribuição | Scaler adotado |
|----------|-------------|---------------|
| `age` | Aprox. normal | StandardScaler |
| `campaign` | Assimétrica + outliers | Log1p → RobustScaler |
| `previous` | Muitos zeros + outliers | RobustScaler |
| Variáveis macroeconômicas | Simétricas, sem extremos | StandardScaler |

### 1.5 — Data Leakage: o vazamento silencioso

**Data leakage** ocorre quando informação que **não estaria disponível em produção** é usada durante o treinamento. O resultado é um modelo com métricas infladas que falha ao ser implantado.

#### O caso `duration` neste dataset

A duração da ligação em segundos tem correlação ≈ 0.40 com o target `y`. Porém:
- Se `duration = 0` → a ligação nem ocorreu → `y = 'no'` com certeza
- A duração **só é conhecida depois que a ligação termina**
- Em produção, ao decidir *para quem ligar*, `duration` simplesmente não existe

> **Decisão:** `duration` é removida **imediatamente** após o carregamento dos dados.

#### Leakage de estatísticas treino/teste

Outro tipo comum: calcular média, desvio ou categorias sobre todo o dataset **antes** do split.

```
ERRADO:  scaler.fit(X_total) → split → transform(X_train), transform(X_test)
         ↑ o scaler "viu" o teste — vazamento!

CORRETO: split → scaler.fit(X_train) → transform(X_train), transform(X_test)
```

O `Pipeline` do scikit-learn, quando usado com `cross_val_score`, aplica essa regra automaticamente.

---
## 🧩 Seção 2 — Estruturas de DataPrep em Python (scikit-learn)

O scikit-learn oferece uma API unificada que garante consistência treino/teste, integração com cross-validation e documentação declarativa do fluxo de transformações.

### 2.1 — Pipeline: encadeamento de transformações

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 2.1 — Pipeline
# ══════════════════════════════════════════════════════
# Um Pipeline encadeia transformadores em sequência.
# O output de cada etapa é o input da seguinte.
# fit() executa fit_transform() em cada passo.

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

np.random.seed(42)
X_demo = pd.DataFrame({
    'idade': [25, 32, np.nan, 45, 28, np.nan, 60],
    'saldo': [1000, 5000, 2000, np.nan, 300, 8000, 1500]
})

# Construção do Pipeline
pipeline_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # passo 1: trata NaN
    ('scaler',  StandardScaler())                   # passo 2: padroniza
])

X_transformado = pipeline_num.fit_transform(X_demo)

print("DataFrame original:")
print(X_demo.to_string())
print()
print("Após Pipeline (imputer + StandardScaler):")
result = pd.DataFrame(X_transformado, columns=X_demo.columns)
print(result.round(3).to_string())
print()
print("Estrutura do pipeline:")
print(pipeline_num)

### 2.2 — ColumnTransformer: tratamentos distintos por grupo de colunas

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 2.2 — ColumnTransformer
# ══════════════════════════════════════════════════════
# Aplica transformadores diferentes a subconjuntos de colunas.
# É o núcleo do nosso preprocessador final.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X_het = pd.DataFrame({
    'idade':    [25, 32, np.nan, 45],
    'saldo':    [1000, 5000, 2000, np.nan],
    'profissao': ['admin', 'operario', np.nan, 'estudante'],
    'educacao': ['medio', 'superior', 'basico', 'superior']
})

pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Cada entrada: (nome, transformador, lista_de_colunas)
preprocessor_demo = ColumnTransformer(transformers=[
    ('numericas',   pipe_num, ['idade', 'saldo']),
    ('categoricas', pipe_cat, ['profissao', 'educacao'])
], remainder='drop')

X_result = preprocessor_demo.fit_transform(X_het)
feature_names = preprocessor_demo.get_feature_names_out()

print("Resultado do ColumnTransformer:")
df_result = pd.DataFrame(X_result, columns=feature_names)
print(df_result.round(3).to_string())
print()
print(f"Shape: {X_result.shape[0]} linhas x {X_result.shape[1]} features geradas")

### 2.3 — Transformadores customizados

Quando as transformações padrão do Sklearn não são suficientes, criamos nossos próprios transformadores. O padrão é herdar de `BaseEstimator` e `TransformerMixin`, o que nos dá `fit_transform()` gratuitamente e compatibilidade total com `Pipeline`.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 2.3 — Transformadores customizados
# ══════════════════════════════════════════════════════
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

# ── Padrão 1: FunctionTransformer (sem estado) ──────────────────────────

log1p_transformer = FunctionTransformer(np.log1p, validate=True, feature_names_out='one-to-one')
# Aplicação: np.log1p(x) = log(1+x), ideal para variáveis com zeros e assimetria positiva


# ── Padrão 2: Classe customizada (com lógica de negócio) ────────────────

class PdaysTransformer(BaseEstimator, TransformerMixin):
    """
    Transforma 'pdays' do dataset Bank Marketing.

    O valor 999 indica que o cliente NAO foi contatado em campanhas
    anteriores — semanticamente diferente de um numero de dias real.

    Gera duas features:
      - pdays_contacted : 1 se contatado antes, 0 caso contrario
      - pdays_value     : dias reais (999 mapeado para 0)
    """

    def fit(self, X, y=None):
        return self  # sem estado para aprender

    def transform(self, X, y=None):
        X_arr = np.array(X).flatten()
        contacted = (X_arr != 999).astype(int)
        value     = np.where(X_arr == 999, 0, X_arr).astype(float)
        return np.column_stack([contacted, value])

    def get_feature_names_out(self, input_features=None):
        return np.array(['pdays_contacted', 'pdays_value'])


# ── Teste rápido ─────────────────────────────────────────────────────────

pdays_sample = np.array([999, 5, 999, 12, 3, 999, 7])
pt = PdaysTransformer()
out = pt.fit_transform(pdays_sample.reshape(-1, 1))
print("PdaysTransformer — resultado:")
print(pd.DataFrame(out, columns=['pdays_contacted', 'pdays_value']).to_string())
print()
print("Transformadores customizados prontos para uso na Pipeline")

### 2.4 — fit, transform, fit_transform: a tríade essencial do Sklearn

```
┌───────────────────┬──────────────────────────────────────────────────────┐
│ Método            │ O que faz                                            │
├───────────────────┼──────────────────────────────────────────────────────┤
│ fit(X)            │ Aprende parâmetros dos dados (média, IQR, categorias) │
│ transform(X)      │ Aplica a transformação usando parâmetros já aprendidos│
│ fit_transform(X)  │ fit + transform em uma chamada — atalho conveniente   │
└───────────────────┴──────────────────────────────────────────────────────┘
```

#### Regra fundamental — nunca violar

```python
# CORRETO
X_train, X_test, y_train, y_test = train_test_split(X, y, ...)

preprocessor.fit(X_train)                    # aprende COM O TREINO apenas
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)  # aplica parâmetros DO TREINO

# ERRADO — leakage de teste
preprocessor.fit(X_test)        # viu o teste!
preprocessor.fit(X)             # viu tudo, incluindo o teste!
```

> 💡 Quando a `Pipeline` é usada com `cross_val_score()`, o scikit-learn garante automaticamente que o `fit` ocorre **apenas** nos folds de treino de cada iteração.

---
## 🔍 Seção 3 — Análise Variável a Variável

Cada variável do dataset recebe análise individual para justificar a técnica escolhida. O objetivo pedagógico é mostrar que **pré-processamento é raciocínio**, não receita.

### 3.0 — Carregamento dos dados

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.0 — Carregamento (Colab ou local)
# ══════════════════════════════════════════════════════

if IN_COLAB:
    # Exibe widget de upload no Colab
    print("Faca o upload do arquivo 'bank-additional-full.csv'")
    uploaded = colab_files.upload()

    if not uploaded:
        raise FileNotFoundError("Nenhum arquivo enviado. Execute a celula novamente.")

    filename = list(uploaded.keys())[0]
    df_raw = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';')
    print(f"Arquivo '{filename}' carregado via upload.")

else:
    # ── Ajuste o caminho conforme necessário ──
    DATA_PATH = 'bank-additional-full.csv'
    df_raw = pd.read_csv(DATA_PATH, sep=';')
    print(f"Arquivo carregado do caminho local: {DATA_PATH}")

print(f"Dimensoes: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas")
df_raw.head(3)

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.0b — Inspeção inicial e mapa de unknowns
# ══════════════════════════════════════════════════════

df = df_raw.copy()

print("Dtypes das colunas:")
print(df.dtypes.to_string())
print()

# Mapa de 'unknown' por coluna categórica
cat_cols = df.select_dtypes(include='object').columns.tolist()
unknown_summary = {}
for col in cat_cols:
    n = (df[col] == 'unknown').sum()
    if n > 0:
        unknown_summary[col] = {'n': n, 'pct': n / len(df) * 100}

if unknown_summary:
    print("Colunas com 'unknown' (missing codificado):")
    print(f"  {'Variavel':<14} {'n':>6}  {'%':>6}  Barra")
    print("  " + "-"*45)
    for col, info in sorted(unknown_summary.items(), key=lambda x: -x[1]['n']):
        bar = chr(9608) * int(info['pct'] / 2)
        print(f"  {col:<14} {info['n']:>6}  {info['pct']:>5.1f}%  {bar}")

### 3.1 — `duration`: remoção por data leakage

> **Decisão: DROP imediato** — nenhum modelo preditivo realista pode usar `duration` como feature.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.1 — Remoção de 'duration' (data leakage)
# ══════════════════════════════════════════════════════

y_binary = (df['y'] == 'yes').astype(int)
corr_duration = df['duration'].corr(y_binary)
zero_dur_count = (df['duration'] == 0).sum()

print(f"Correlacao de Pearson: duration x y = {corr_duration:.4f}")
print(f"Registros com duration=0: {zero_dur_count} (100% deles tem y='no')")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma por classe
for label, grp in df.groupby('y')['duration']:
    axes[0].hist(grp, bins=60, alpha=0.65, label=label, density=True)
axes[0].set_title('Distribuicao de duration por classe')
axes[0].set_xlabel('Duracao (segundos)')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# Barras: duration=0 vs duration>0
n_zero = zero_dur_count
n_pos  = len(df) - n_zero
bars = axes[1].bar(['duration = 0', 'duration > 0'], [n_zero, n_pos],
                   color=['#e74c3c', '#3498db'], alpha=0.85, edgecolor='white')
axes[1].set_title(f'duration=0 -> 100% "no" ({n_zero} casos)')
axes[1].set_ylabel('Contagem')
for bar, val in zip(bars, [n_zero, n_pos]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.suptitle("Evidencia de data leakage em 'duration'", fontsize=13)
plt.tight_layout()
plt.show()

# Remove a coluna
df = df.drop(columns=['duration'])
print(f"'duration' removida. Shape atual: {df.shape}")

### 3.2 — `pdays`: transformação especial com transformador customizado

O valor `999` não é um número de dias — é uma convenção semântica que indica *"cliente não foi contatado em campanhas anteriores"*. Tratar 999 como um valor numérico real distorceria qualquer scaler e introduziria relações falsas no modelo.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.2 — Transformação de 'pdays'
# ══════════════════════════════════════════════════════

n_999 = (df['pdays'] == 999).sum()
pct_999 = n_999 / len(df)

print(f"Registros com pdays=999 (nao contatado): {n_999:,} ({pct_999:.1%})")
print(f"Range dos valores reais (excl. 999): "
      f"{df[df['pdays']!=999]['pdays'].min()} - {df[df['pdays']!=999]['pdays'].max()}")

# Taxa de conversao por grupo
contacted_mask = df['pdays'] != 999
conv_contato    = df[contacted_mask]['y'].eq('yes').mean()
conv_sem_contato = df[~contacted_mask]['y'].eq('yes').mean()
print()
print(f"Taxa conversao — com contato anterior : {conv_contato:.2%}")
print(f"Taxa conversao — sem contato anterior : {conv_sem_contato:.2%}")
print(f"Razao: {conv_contato/conv_sem_contato:.1f}x maior para quem ja foi contatado")

# Aplica o PdaysTransformer definido na Secao 2.3
pt = PdaysTransformer()
pdays_out = pt.fit_transform(df[['pdays']].values)

df['pdays_contacted'] = pdays_out[:, 0].astype(int)
df['pdays_value']     = pdays_out[:, 1].astype(float)
df = df.drop(columns=['pdays'])

print()
print("pdays transformado em duas features:")
print(df[['pdays_contacted', 'pdays_value']].describe().T.round(2).to_string())

### 3.3 — Variáveis com 'unknown': diagnóstico e tratamento diferenciado

Nem todos os 'unknowns' são iguais. A estratégia de tratamento depende da proporção e, principalmente, do **mecanismo de ausência** hipotético.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.3 — Diagnóstico e tratamento dos 'unknowns'
# ══════════════════════════════════════════════════════

# Analise comparativa: taxa de conversao dos unknowns vs. categorias reais
unknown_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']

print("Taxa de conversao (y=yes) para 'unknown' vs. outras categorias:")
print(f"  {'Variavel':<12}  {'conv unknown':>14}  {'conv demais':>12}  {'% unknown':>10}  Estrategia")
print("  " + "-"*70)

for col in unknown_cols:
    pct_unk = (df[col] == 'unknown').mean()
    mask_unk = df[col] == 'unknown'
    conv_unk  = df[mask_unk]['y'].eq('yes').mean() if mask_unk.sum() > 0 else float('nan')
    conv_rest = df[~mask_unk]['y'].eq('yes').mean()
    diff = abs(conv_unk - conv_rest) if not np.isnan(conv_unk) else 0

    if col == 'default':
        strategy = "Manter como categoria (MNAR, ~20%)"
    else:
        strategy = "Imputar pela moda"

    print(f"  {col:<12}  {conv_unk:>13.2%}  {conv_rest:>11.2%}  "
          f"{pct_unk:>9.1%}  {strategy}")

print()
print("Decisao:")
print("  - 'default': 20.9% unknown + diferenca de conversao => MNAR provavel => manter")
print("  - Demais: proporcao baixa + distribuicao aleatoria => imputar pela moda")

# Aplica a decisao: converte 'unknown' para NaN nas variaveis que serao imputadas
impute_with_mode = ['job', 'marital', 'education', 'housing', 'loan']
for col in impute_with_mode:
    df[col] = df[col].replace('unknown', np.nan)

print()
print("'unknown' convertido para NaN em:", impute_with_mode)
print("'default': mantido com 'unknown' como categoria propria")
print(f"Shape apos tratamento: {df.shape}")

### 3.4 — `education`: ordinal encoding com hierarquia explícita

A variável `education` tem uma progressão natural e mensurável: mais anos de estudo corresponde a nível mais elevado. Ignorar essa ordem (usando OHE) desperdiçaria uma relação real que o modelo poderia aprender.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.4 — Análise de 'education' e ordinal encoding
# ══════════════════════════════════════════════════════

# Hierarquia educacional definida explicitamente
EDUCATION_ORDER = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
                   'high.school', 'professional.course', 'university.degree',
                   np.nan]  # NaN (ex-unknown) ao final

edu_labels = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
              'high.school', 'professional.course', 'university.degree']

present_labels = [c for c in edu_labels if c in df['education'].values or
                  (c in df['education'].dropna().values)]

conv_rates = []
counts     = []
for cat in present_labels:
    subset = df[df['education'] == cat]
    conv_rates.append(subset['y'].eq('yes').mean() if len(subset) > 0 else 0)
    counts.append(len(subset))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Taxa de conversao por nivel
colors = plt.cm.RdYlGn([i / max(len(present_labels)-1, 1) for i in range(len(present_labels))])
bars = axes[0].bar(range(len(present_labels)), conv_rates, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xticks(range(len(present_labels)))
axes[0].set_xticklabels(present_labels, rotation=35, ha='right', fontsize=9)
axes[0].set_title("Taxa de conversao por nivel de educacao")
axes[0].set_ylabel("Taxa de conversao (y=yes)")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
for b, v in zip(bars, conv_rates):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.002,
                 f'{v:.1%}', ha='center', va='bottom', fontsize=8)

# Contagem por nivel
axes[1].bar(range(len(present_labels)), counts, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xticks(range(len(present_labels)))
axes[1].set_xticklabels(present_labels, rotation=35, ha='right', fontsize=9)
axes[1].set_title("Volume por nivel de educacao")
axes[1].set_ylabel("Contagem de registros")

plt.suptitle("Analise de 'education' — justificativa para OrdinalEncoder", fontsize=13)
plt.tight_layout()
plt.show()

print("Conclusao: ha tendencia crescente de conversao com escolaridade.")
print("OrdinalEncoder com ordem explicita e a escolha tecnica correta.")
print("OHE ignoraria essa relacao de ordenacao.")

### 3.5 — `month` e `day_of_week`: cyclical encoding

O OHE trataria janeiro e dezembro como categorias completamente independentes — mas temporalmente são adjacentes. O encoding sin/cos transforma meses e dias em coordenadas em um círculo, preservando a continuidade.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.5 — Cyclical encoding: month e day_of_week
# ══════════════════════════════════════════════════════

MONTH_MAP = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
DAY_MAP   = {'mon':1,'tue':2,'wed':3,'thu':4,'fri':5}

df['month_num'] = df['month'].map(MONTH_MAP)
df['day_num']   = df['day_of_week'].map(DAY_MAP)

# Encoding sin/cos
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)
df['day_sin']   = np.sin(2 * np.pi * df['day_num'] / 5)
df['day_cos']   = np.cos(2 * np.pi * df['day_num'] / 5)

df = df.drop(columns=['month', 'day_of_week', 'month_num', 'day_num'])

# Visualizacao: meses e dias no espaco ciclico
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theta_m = np.linspace(0, 2*np.pi, 12, endpoint=False)
month_names = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
xm = np.cos(theta_m); ym = np.sin(theta_m)
sc = axes[0].scatter(xm, ym, c=range(12), cmap='hsv', s=180, zorder=3)
for i, name in enumerate(month_names):
    axes[0].annotate(name, (xm[i]*1.22, ym[i]*1.22), ha='center', va='center', fontsize=9)
axes[0].add_patch(plt.Circle((0,0), 1, fill=False, linestyle='--', alpha=0.25))
axes[0].set_xlim(-1.5, 1.5); axes[0].set_ylim(-1.5, 1.5)
axes[0].set_aspect('equal')
axes[0].set_title('Meses no espaco ciclico (cos, sin)\nDez <-> Jan: distancia pequena')
axes[0].axhline(0, color='gray', alpha=0.3); axes[0].axvline(0, color='gray', alpha=0.3)
axes[0].set_xticks([]); axes[0].set_yticks([])

theta_d = np.linspace(0, 2*np.pi, 5, endpoint=False)
day_names = ['Seg','Ter','Qua','Qui','Sex']
xd = np.cos(theta_d); yd = np.sin(theta_d)
axes[1].scatter(xd, yd, c=range(5), cmap='cool', s=180, zorder=3)
for i, name in enumerate(day_names):
    axes[1].annotate(name, (xd[i]*1.25, yd[i]*1.25), ha='center', va='center', fontsize=10)
axes[1].add_patch(plt.Circle((0,0), 1, fill=False, linestyle='--', alpha=0.25))
axes[1].set_xlim(-1.6, 1.6); axes[1].set_ylim(-1.6, 1.6)
axes[1].set_aspect('equal')
axes[1].set_title('Dias da semana no espaco ciclico')
axes[1].axhline(0, color='gray', alpha=0.3); axes[1].axvline(0, color='gray', alpha=0.3)
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.suptitle("Cyclical encoding: continuidade circular preservada", fontsize=13)
plt.tight_layout()
plt.show()

print("Geradas: month_sin, month_cos, day_sin, day_cos")
print("Removidas: month, day_of_week")

### 3.6 — Variáveis numéricas: distribuição e escolha do scaler

Antes de escolher o scaler, precisamos entender a distribuição de cada variável: simetria, presença de outliers e magnitude dos valores.

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.6 — Análise de distribuição das numéricas
# ══════════════════════════════════════════════════════

num_cols = ['age', 'campaign', 'previous', 'pdays_value',
            'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
            'euribor3m', 'nr.employed']

# Estatisticas de assimetria e outliers via IQR
rows = []
for col in num_cols:
    if col not in df.columns:
        continue
    s      = df[col].skew()
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr    = q3 - q1
    out_lo = df[col] < q1 - 1.5 * iqr
    out_hi = df[col] > q3 + 1.5 * iqr
    out_pct = (out_lo | out_hi).mean()

    if col == 'campaign':
        rec = 'Log1p + RobustScaler'
    elif out_pct > 0.04 or abs(s) > 1.5:
        rec = 'RobustScaler'
    else:
        rec = 'StandardScaler'

    rows.append({'Variavel': col,
                 'Skewness': round(s, 2),
                 'Outliers%': f'{out_pct:.1%}',
                 'Scaler': rec})

stats_df = pd.DataFrame(rows)
print("Analise de distribuicao e scaler recomendado:")
print(stats_df.to_string(index=False))

# Visualizacao das distribuicoes
present_cols = [c for c in num_cols if c in df.columns]
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(present_cols[:9]):
    axes[i].hist(df[col], bins=50, color='steelblue', alpha=0.75, edgecolor='white')
    axes[i].set_title(f"{col}  (skew={df[col].skew():.2f})", fontsize=10)
    axes[i].tick_params(labelsize=8)

for j in range(len(present_cols), 9):
    axes[j].set_visible(False)

plt.suptitle('Distribuicoes das variaveis numericas', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3.7 — Demais variáveis categóricas e encoding do target `y`

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 3.7 — Categóricas restantes e target
# ══════════════════════════════════════════════════════

# Inspeciona categorias restantes
remaining_cat = [c for c in df.select_dtypes('object').columns if c != 'y']
print("Variaveis categoricas para OHE:")
for col in remaining_cat:
    cats = sorted(df[col].dropna().unique())
    pct_na = df[col].isna().mean()
    print(f"  {col:<12} {len(cats)+1 if pct_na>0 else len(cats):>2} cat  "
          f"NaN={pct_na:.1%}  {cats}")

# Distribuicao do target
print()
print("Distribuicao do target 'y':")
vc = df['y'].value_counts()
for val, cnt in vc.items():
    pct = cnt / len(df)
    bar = chr(9608) * int(pct / 0.02)
    print(f"  {val:<5} {cnt:>6,} ({pct:.1%})  {bar}")

ratio = vc.get('no', 0) / vc.get('yes', 1)
print()
print(f"Desbalanceamento: 1 : {ratio:.0f}")
print("IMPORTANTE: classe minoritaria (yes) sera tratada na modelagem com")
print("  class_weight='balanced' ou tecnicas de oversampling (SMOTE).")

# Encoding do target: yes -> 1, no -> 0
df['y_encoded'] = (df['y'] == 'yes').astype(int)
df = df.drop(columns=['y'])

print()
print(f"Target codificado: yes=1, no=0")
print(f"Shape final do dataframe preparado: {df.shape}")

---
## ⚙️ Seção 4 — Geração do Dataset Final

### 4.1 — Arquitetura da Pipeline completa

```
                    ┌──────────────────────────────────────────────┐
                    │            ColumnTransformer                  │
                    │                                              │
   ┌────────────────┼───────────────────────────────────────────┐  │
   │ num_std        │ StandardScaler                            │  │
   │                │ age, emp.var.rate, cons.price.idx,        │  │
   │                │ cons.conf.idx, euribor3m, nr.employed     │  │
   ├────────────────┼───────────────────────────────────────────┤  │
   │ num_robust     │ RobustScaler                              │  │
   │                │ previous, pdays_value                     │  │
   ├────────────────┼───────────────────────────────────────────┤  │
   │ num_log_robust │ Log1p → RobustScaler                      │  │
   │                │ campaign                                   │  │
   ├────────────────┼───────────────────────────────────────────┤  │
   │ education      │ SimpleImputer(moda) → OrdinalEncoder      │  │
   ├────────────────┼───────────────────────────────────────────┤  │
   │ ohe_impute     │ SimpleImputer(moda) → OHE(drop='first')  │  │
   │                │ job, marital, housing, loan                │  │
   ├────────────────┼───────────────────────────────────────────┤  │
   │ ohe_no_impute  │ OHE(drop='first')                         │  │
   │                │ default, contact, poutcome                 │  │
   ├────────────────┼───────────────────────────────────────────┤  │
   │ passthrough    │ sem transformacao                          │  │
   │                │ pdays_contacted, month_sin/cos, day_sin/cos│  │
   └────────────────┴───────────────────────────────────────────┘  │
                    └──────────────────────────────────────────────┘
```

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 4.2 — Train/Test Split estratificado
# ══════════════════════════════════════════════════════

y = df['y_encoded'].values
X = df.drop(columns=['y_encoded'])

print(f"Features disponíveis: {X.shape[1]} colunas")
print(f"Registros: {X.shape[0]:,}")
print(f"Positivos (y=1): {y.sum():,} ({y.mean():.2%})")
print()

# Split estratificado — preserva proporcao de classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(f"Treino : {X_train.shape[0]:,} registros | positivos: {y_train.mean():.2%}")
print(f"Teste  : {X_test.shape[0]:,} registros  | positivos: {y_test.mean():.2%}")
print()
print("Split estratificado OK — proporcao de classes preservada")

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 4.3 — Construção e execução da Pipeline completa
# ══════════════════════════════════════════════════════

# ── Grupos de colunas ────────────────────────────────────────────────────

NUM_STD_COLS        = ['age', 'emp.var.rate', 'cons.price.idx',
                       'cons.conf.idx', 'euribor3m', 'nr.employed']
NUM_ROBUST_COLS     = ['previous', 'pdays_value']
NUM_LOG_ROBUST_COLS = ['campaign']
EDUCATION_COLS      = ['education']
OHE_IMPUTE_COLS     = ['job', 'marital', 'housing', 'loan']
OHE_NO_IMPUTE_COLS  = ['default', 'contact', 'poutcome']
PASSTHROUGH_COLS    = ['pdays_contacted', 'month_sin', 'month_cos',
                       'day_sin', 'day_cos']

# ── Pipelines por ramo ──────────────────────────────────────────────────

pipe_num_std = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipe_num_robust = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])

pipe_num_log_robust = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log',     FunctionTransformer(np.log1p, validate=True, feature_names_out='one-to-one')),
    ('scaler',  RobustScaler())
])

EDUCATION_ORDER = [['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
                    'high.school', 'professional.course',
                    'university.degree', 'unknown']]

pipe_education = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=EDUCATION_ORDER,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

pipe_ohe_impute = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False,
        drop='first'
    ))
])

pipe_ohe_no_impute = Pipeline([
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False,
        drop='first'
    ))
])

# ── ColumnTransformer principal ─────────────────────────────────────────

preprocessor = ColumnTransformer(
    transformers=[
        ('num_std',        pipe_num_std,        NUM_STD_COLS),
        ('num_robust',     pipe_num_robust,     NUM_ROBUST_COLS),
        ('num_log_robust', pipe_num_log_robust, NUM_LOG_ROBUST_COLS),
        ('education',      pipe_education,      EDUCATION_COLS),
        ('ohe_impute',     pipe_ohe_impute,     OHE_IMPUTE_COLS),
        ('ohe_no_impute',  pipe_ohe_no_impute,  OHE_NO_IMPUTE_COLS),
        ('passthrough',    'passthrough',        PASSTHROUGH_COLS),
    ],
    remainder='drop',
    verbose_feature_names_out=True
)

# ── Fit no treino → Transform em treino e teste ──────────────────────────

print("Ajustando preprocessador no conjunto de treino...")
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

feature_names_out = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_proc, columns=feature_names_out)
X_test_df  = pd.DataFrame(X_test_proc,  columns=feature_names_out)

print(f"Preprocessing concluido!")
print(f"  Treino processado : {X_train_df.shape}")
print(f"  Teste  processado : {X_test_df.shape}")
print(f"  Features geradas  : {X_train_df.shape[1]}")
print()
print("Primeiras 10 features geradas:")
for i, name in enumerate(feature_names_out[:10]):
    print(f"  {i+1:>2}. {name}")

In [ ]:
# ══════════════════════════════════════════════════════
# SEÇÃO 4.4 — Validação e exportação do dataset final
# ══════════════════════════════════════════════════════

print("Checks de qualidade:")
print("-" * 55)

# 1. Sem valores nulos
null_tr = X_train_df.isnull().sum().sum()
null_te = X_test_df.isnull().sum().sum()
status1 = "OK" if null_tr == 0 and null_te == 0 else f"ERRO: {null_tr+null_te} nulos"
print(f"  Valores nulos             : {status1}")

# 2. Shape consistente
status2 = "OK" if X_train_df.shape[1] == X_test_df.shape[1] else "ERRO: shapes diferentes"
print(f"  Consistencia de features  : {status2}")

# 3. Distribuicao do target preservada
diff_target = abs(y_train.mean() - y_test.mean())
status3 = f"OK (diferenca={diff_target:.4f})" if diff_target < 0.01 else f"ATENCAO diff={diff_target:.4f}"
print(f"  Balanco do target         : {status3}")

# 4. Ranges apos normalizacao
max_abs_train = np.abs(X_train_df.values).max()
status4 = "OK" if max_abs_train < 100 else f"Verificar (max={max_abs_train:.1f})"
print(f"  Magnitude dos valores     : {status4}")

print()
print("Estatisticas descritivas do treino (amostra — 6 primeiras features):")
print(X_train_df.describe().round(3).iloc[:, :6].to_string())

# ── Exportacao ───────────────────────────────────────────────────────────

train_export = X_train_df.copy()
train_export['y'] = y_train

test_export = X_test_df.copy()
test_export['y'] = y_test

train_export.to_csv('bank_preprocessed_train.csv', index=False)
test_export.to_csv('bank_preprocessed_test.csv',  index=False)

print()
print("Arquivos exportados:")
print(f"  bank_preprocessed_train.csv — {train_export.shape[0]:,} linhas x {train_export.shape[1]} colunas")
print(f"  bank_preprocessed_test.csv  — {test_export.shape[0]:,} linhas x {test_export.shape[1]} colunas")

# Download automatico no Colab
if IN_COLAB:
    print()
    print("Iniciando download no Colab...")
    colab_files.download('bank_preprocessed_train.csv')
    colab_files.download('bank_preprocessed_test.csv')

print()
print("Pipeline de pre-processamento concluida com sucesso!")

---
## 📋 Seção 5 — Conclusões

### 5.1 — Tabela-resumo das decisões de pré-processamento

| Variável | Tipo | Missing | Tratamento de missing | Encoding / Scaling | Justificativa |
|----------|------|---------|----------------------|--------------------|---------------|
| `duration` | Numérica | — | — | **DROP** | Data leakage: disponível apenas após a ligação |
| `pdays` | Especial | — | — | Custom → 2 features | 999 = semântica própria ("não contatado") |
| `age` | Numérica | Não | — | StandardScaler | Distribuição próxima de normal |
| `campaign` | Numérica | Não | — | Log1p + RobustScaler | Assimétrica com outliers legítimos |
| `previous` | Numérica | Não | — | RobustScaler | Muitos zeros + outliers |
| Vars. macro | Numéricas | Não | — | StandardScaler | Distribuições simétricas |
| `month` | Temporal | Não | — | sin/cos cíclico | Periodicidade: dez ↔ jan |
| `day_of_week` | Temporal | Não | — | sin/cos cíclico | Periodicidade: sex ↔ seg |
| `education` | Ordinal | ~4% | Moda | OrdinalEncoder (ordem explícita) | Hierarquia educacional real |
| `default` | Nominal | ~21% | **Manter 'unknown'** | OHE | MNAR provável: inadimplência omitida intencionalmente |
| `job`, `marital` | Nominal | <1% | Moda | OHE (drop='first') | Proporção baixa, sem ordem natural |
| `housing`, `loan` | Nominal | ~2% | Moda | OHE (drop='first') | Idem |
| `contact`, `poutcome` | Nominal | Não | — | OHE (drop='first') | Nominal, sem missing |
| `y` | Target | — | — | LabelEncoder (yes→1) | Classificação binária |

### 5.2 — Reflexões pedagógicas e próximos passos

#### Lições-chave deste notebook

**1. Pré-processamento é raciocínio, não receita.**
Cada variável recebeu tratamento específico baseado em análise da distribuição, mecanismo de ausência e semântica de domínio. Uma abordagem única para todas as variáveis levaria a erros.

**2. A ordem das operações é crítica.**
O split treino/teste **precisa** anteceder o `fit` de qualquer transformador. Inverter essa ordem contamina o modelo com informação do teste e produz métricas falsamente otimistas.

**3. Dados faltantes exigem hipótese.**
`default='unknown'` e `job='unknown'` recebem tratamentos diferentes porque os mecanismos de ausência são diferentes — não é só a proporção que importa, mas o **porquê** do dado estar ausente.

**4. Transformadores customizados são cidadãos de primeira classe.**
`PdaysTransformer` herda de `BaseEstimator` e `TransformerMixin` e se encaixa perfeitamente na `Pipeline`, garantindo o mesmo contrato de `fit/transform` que os transformadores nativos.

---

#### Próximas etapas

| Fase | Técnica | Por quê é necessário |
|------|---------|----------------------|
| **Desbalanceamento** | SMOTE, ADASYN, `class_weight='balanced'` | Ratio ≈ 1:8 distorce modelos que otimizam acurácia global |
| **Seleção de features** | RFE, Permutation Importance, SHAP | Pipeline gerou 30+ features; algumas podem ser redundantes |
| **Baseline** | Regressão Logística, Árvore de Decisão | Sempre comece simples para ter referência de comparação |
| **Modelos avançados** | Random Forest, XGBoost, LightGBM | Candidatos naturais para dados tabulares desbalanceados |
| **Avaliação** | AUC-ROC, AUC-PR, F1 macro | Accuracy é enganosa com desbalanceamento — não usar como métrica principal |
| **Interpretabilidade** | SHAP values | Essencial para decisões operacionais de call center |

> 💡 **Insight de negócio:** Para priorização de filas de call center, a métrica mais relevante é a **AUC-PR** (Área sob a Curva Precision-Recall), que mede diretamente a qualidade do ranking de clientes com maior propensão à adesão — independente do limiar de classificação escolhido.